# Day 2 — Testing the Voice Agent with an LLM Judge (DeepEval)

**Module 9 · Voice Agent Testing**

---

Day 1 built the agent and *introduced* the tests. Today we **run** them — three with **DeepEval `GEval`** (the LLM-as-judge from Modules 4, 7 & 8, now on voice), plus a latency check that needs no judge.

| # | Test | Stage | How |
|---|---|---|---|
| 1 | **Reply quality** — relevant, safe, speakable | LLM (brain) | `GEval` judge |
| 2 | **STT fidelity** — did the ears hear it right? | STT (ears) | `GEval` judge |
| 3 | **TTS intelligibility** — did the mouth say it right? (round-trip) | TTS (mouth) | `GEval` judge |
| 4 | **Latency** — is a turn fast enough to feel live? | end-to-end | plain `timings_ms` budget |

> **Live notebook** — needs `SARVAM_API_KEY` + `GROQ_API_KEY` in `.env`, and `pip install -r ../requirements.txt` (now includes `deepeval`).

## The judge

We grade **meaning, not exact strings** (speech is noisy — Day 1's agent heard "Manali" as "Lanali"). The judge is a **DeepEval `GEval`** metric backed by a Groq model via DeepEval's `LocalModel` (OpenAI-compatible), so it's self-contained with the key you already have.

Good practice: the judge is a **stronger, different model** (`llama-3.3-70b-versatile`) than the agent's brain (`llama-3.1-8b-instant`) — you don't want a model grading its own homework.

## How do you test *speech*? Transcription is the bridge

A test needs something it can compare — a string, a number, a rubric. But speech is a **waveform**: you can't write `assert audio == expected`, and the LLM judge can't "listen" to a `.wav`. So we never test the audio directly. We turn speech into **text** with **transcription (STT)**, then grade the text with the tools we already have.

That means **transcription is the bridge that makes every voice test possible** — and it plays a *different role* in each test:

| Test | What we compare (all text!) | Transcription's role |
|---|---|---|
| **1 · Reply quality** (brain) | the reply text vs. the question | already done *upstream* — the brain answered the **transcript**, so we just grade that text |
| **2 · STT fidelity** (ears) | transcript **vs. the sentence we know we said** | transcription **is the thing under test** — if the transcript's meaning drifts, the ears failed |
| **3 · TTS intelligibility** (mouth) | speech transcribed-back **vs. the line we asked it to say** | transcription is the **measuring instrument** — we can't listen in code, so STT "listens" for us (a round-trip) |

> **Honest caveat:** in Test 3 we use the *ears* (STT) to judge the *mouth* (TTS), so a failure could be either one. We assume STT is trustworthy — which is exactly why we test it separately in Test 2. When a round-trip fails, play the clip yourself to see whether the mouth garbled it or the ears misheard.

In [1]:
import os, sys, warnings
warnings.filterwarnings("ignore")            # hide DeepEval's deprecation notices
os.environ["DEEPEVAL_TELEMETRY_OPT_OUT"] = "1"

sys.path.insert(0, ".")
from voice_agent import VoiceAgent
from deepeval.models import LocalModel
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams as P

# The judge: DeepEval GEval, backed by Groq (stronger model than the agent uses).
judge = LocalModel(
    model="llama-3.3-70b-versatile",
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1",
)

agent = VoiceAgent()

def grade(metric, test_case):
    """Run one GEval metric and print its verdict."""
    metric.measure(test_case)
    print(f"[{'PASS' if metric.is_successful() else 'FAIL'}]  {metric.name}: {metric.score:.2f}")
    print(f"       why: {metric.reason}\n")

print("Judge ready:", judge.get_model_name())

Judge ready: llama-3.3-70b-versatile (Local Model)


## One real turn to grade

Run the agent once; we'll grade its transcript and reply. `reference` is the exact sentence we synthesized, so we know the truth to compare the STT against.

In [2]:
reference = "What should I pack for a trip to Manali in December?"
question_wav = agent.make_audio(reference, "audio/q.wav")
turn = agent.respond(question_wav, out_path="audio/reply.wav")

print("reference :", reference)
print("heard     :", turn.transcript)     # STT output
print("reply     :", turn.reply_text)     # LLM output
print("timings   :", turn.timings_ms, "ms")

reference : What should I pack for a trip to Manali in December?
heard     : That should be packed for a trip to Lanali in December.
reply     : You'll want to pack warm clothing for Lanali in December, as it's likely to be quite chilly.
timings   : {'stt': 1016, 'llm': 492, 'llm_first_token': 485, 'tts': 724, 'total': 2232} ms


## Test 1 — Reply quality (the brain)

Is the reply relevant, safe, and **speakable**? This is the same idea as Module 4's `GEval` and Module 8's `llm-rubric`, on the transcript→reply step.

In [3]:
reply_quality = GEval(
    name="Reply quality",
    criteria="The reply directly answers the question, is safe, and is speakable: "
             "1-3 short sentences with no markdown, bullet lists, or symbols.",
    evaluation_params=[P.INPUT, P.ACTUAL_OUTPUT],
    model=judge, threshold=0.7,
    async_mode=False,   # grade synchronously — robust inside Jupyter (avoids a nest_asyncio loop error)
)
grade(reply_quality, LLMTestCase(input=turn.transcript, actual_output=turn.reply_text))

[PASS]  Reply quality: 1.00
       why: The Actual Output directly answers the question posed by the Input, providing relevant information about packing for a trip to Lanali in December, and is safe, speakable, and concise, making it a suitable response.



## Test 2 — STT fidelity (the ears)

Did the transcript preserve the *meaning* of what we actually said? We compare the transcript (`actual_output`) to the known `reference` (`expected_output`). A changed place name or number should **fail** — this is the check that catches the "Manali → Lanali" slip from Day 1.

In [4]:
stt_fidelity = GEval(
    name="STT fidelity",
    criteria="The actual transcript preserves the meaning of the expected sentence. "
             "Minor wording differences are fine, but a changed place name or number is a failure.",
    evaluation_params=[P.EXPECTED_OUTPUT, P.ACTUAL_OUTPUT],
    model=judge, threshold=0.7,
    async_mode=False,
)
grade(stt_fidelity, LLMTestCase(input=reference, actual_output=turn.transcript, expected_output=reference))

[FAIL]  STT fidelity: 0.00
       why: The actual transcript contains significant differences from the expected sentence, including a change in the verb and a misspelling of the place name 'Manali' as 'Lanali', which alters the original meaning and fails to accurately replicate key details



## Test 3 — TTS intelligibility (the mouth)

Can the agent's **speech** be understood? We can't "listen" in a test, so we do a **round-trip**: speak a line, transcribe it back, and judge whether the meaning — especially the number and the city — survived.

In [5]:
spoken_line = "Your order 4728 will arrive in Chennai on Friday."
clip = agent.make_audio(spoken_line, "audio/tts_check.wav")
heard_back, _, _ = agent.transcribe(clip)
print("spoke:", spoken_line)
print("heard:", heard_back, "\n")

tts_intelligibility = GEval(
    name="TTS intelligibility",
    criteria="The actual (heard-back) text preserves the meaning of the expected spoken line, "
             "including the order number and the city. A changed number or place is a failure.",
    evaluation_params=[P.EXPECTED_OUTPUT, P.ACTUAL_OUTPUT],
    model=judge, threshold=0.7,
    async_mode=False,
)
grade(tts_intelligibility, LLMTestCase(input=spoken_line, actual_output=heard_back, expected_output=spoken_line))

spoke: Your order 4728 will arrive in Chennai on Friday.
heard: Your order 4728 will arrive in Hanoi on Friday. 



[FAIL]  TTS intelligibility: 0.00
       why: The city in the Expected Output and Actual Output do not match, with 'Chennai' expected but 'Hanoi' given, altering the original intent and information



## Test 4 — Latency (end-to-end)

Not everything needs a judge. Latency is a plain number the agent already gives us (Module 4 Day 5's `LatencyMetric`, on voice). A turn much over ~2s feels laggy in conversation — and the per-stage breakdown tells you *which* organ (ears / brain / mouth) to optimize first.

In [6]:
BUDGET_MS = 8000   # generous ceiling for a demo; tighten toward what you actually see
t = turn.timings_ms
print(f"[{'PASS' if t['total'] < BUDGET_MS else 'FAIL'}]  total turn: {t['total']} ms  (budget {BUDGET_MS} ms)")
print("       breakdown:", {k: t[k] for k in ("stt", "llm", "tts")}, "ms")

# Which stage is slowest? That's where to optimize first.
slowest = max(("stt", "llm", "tts"), key=lambda k: t[k])
print(f"       slowest stage: {slowest} ({t[slowest]} ms)")

[PASS]  total turn: 2232 ms  (budget 8000 ms)
       breakdown: {'stt': 1016, 'llm': 492, 'tts': 724} ms
       slowest stage: stt (1016 ms)


## Summary

- We graded all three stages with **one tool** — DeepEval `GEval`, judged by Groq — because voice needs **meaning-level** grading, not string matching.
  - **Reply quality** (brain) · **STT fidelity** (ears) · **TTS intelligibility** (mouth, via round-trip)
- **Latency** stayed a plain numeric check on `timings_ms` — no judge needed.
- Each `GEval` is just a plain-English **criteria** + which fields the judge sees — the same pattern as Modules 4/8, now on audio.

**Extend it:** add more turns (accents, code-switching, numbers) and reuse these exact metrics; add `jiwer` WER alongside STT fidelity for hard cases; tighten the latency budget. That's the full pipeline-stage × failure-mode coverage — the Module 4 Day 4 mindset, one last time, for voice.

That closes the course: from `assert` in Module 2 to a voice agent you can **build and trust** — the same testing mindset, one new surface at a time.